# 03 — Agent Anatomy & The PRAM Loop (Code Examples)

Build a minimal PRAM loop from scratch and visualize each phase.

**Prerequisites:** `pip install openai python-dotenv rich`

In [ ]:
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
from rich import print as rprint
from rich.panel import Panel
from rich.console import Console

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
console = Console()
print('Setup complete')

## Phase P: Perception — Context Assembly

In [ ]:
class PerceptionModule:
    """
    PERCEPTION: Assembles the context window from all available inputs.
    This is what gets sent to the LLM for reasoning.
    """
    def __init__(self, system_prompt: str):
        self.system_prompt = system_prompt

    def assemble_context(self, 
                         user_input: str, 
                         history: list[dict],
                         tool_result: str | None = None) -> list[dict]:
        """Assembles all inputs into a single ordered context."""
        
        messages = [{"role": "system", "content": self.system_prompt}]
        messages.extend(history)  # Past conversation
        messages.append({"role": "user", "content": user_input})  # Current input
        
        total_chars = sum(len(str(m)) for m in messages)
        print(f"PERCEPTION: Context assembled")
        print(f"  Components: system prompt + {len(history)} history msgs + user input")
        print(f"  Total context size: ~{total_chars} chars (~{total_chars//4} tokens)")
        return messages

# Demonstrate perception
perception = PerceptionModule(
    system_prompt="You are an AI agent that helps with research tasks. Use available tools when needed."
)

mock_history = [
    {"role": "user",      "content": "My research topic is AI agents."},
    {"role": "assistant", "content": "Great! I can help you research AI agents."},
]

context = perception.assemble_context(
    user_input="Find the key agentic design patterns.",
    history=mock_history
)

print(f"\nContext window structure:")
for i, msg in enumerate(context):
    print(f"  [{i}] role={msg['role']:10} chars={len(msg['content'])}")

## Phase R: Reasoning — The LLM Brain

In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search the web for current information.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Search query"}},
            "required": ["query"]
        }
    }
}]

class ReasoningModule:
    """
    REASONING: The LLM processes context and outputs either:
    1. A tool call (ACTION decision), or
    2. A final answer (TERMINATE decision)
    """
    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.total_tokens = 0

    def reason(self, messages: list[dict]) -> dict:
        """LLM reasoning: returns decision with type 'tool_call' or 'final_answer'."""
        
        response = client.chat.completions.create(
            model=self.model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        self.total_tokens += response.usage.total_tokens
        
        if msg.tool_calls:
            tool_call = msg.tool_calls[0]
            decision = {
                "type": "tool_call",
                "tool": tool_call.function.name,
                "args": json.loads(tool_call.function.arguments),
                "tool_call_id": tool_call.id,
                "raw_message": msg
            }
            print(f"REASONING: Decided to call tool '{decision['tool']}'")
            print(f"           Args: {decision['args']}")
        else:
            decision = {
                "type": "final_answer",
                "content": msg.content,
                "raw_message": msg
            }
            print(f"REASONING: Generated final answer ({len(msg.content)} chars)")
        
        print(f"           Tokens used this step: {response.usage.total_tokens}")
        return decision

# Test reasoning
reasoning = ReasoningModule()
test_messages = [
    {"role": "system", "content": "You are an AI agent. Use web_search for current data."},
    {"role": "user",   "content": "What are the top 3 AI agent frameworks in 2025?"}
]

decision = reasoning.reason(test_messages)
print(f"\nDecision type: {decision['type']}")

## Phase A: Action — Tool Execution

In [ ]:
class ActionModule:
    """
    ACTION: Executes the tool the reasoning module selected.
    Returns structured observations back to the agent loop.
    """
    def __init__(self):
        # Tool registry: map tool names to functions
        self.registry = {
            "web_search": self._web_search,
            "calculator": self._calculator,
        }
    
    def _web_search(self, query: str) -> str:
        """Mock web search — replace with Tavily/SerpAPI in production."""
        responses = {
            "ai agent frameworks 2025": "Top frameworks: LangGraph, CrewAI, AutoGen, Agno, LangFlow",
            "default": f"Search results for '{query}': [mock data]"
        }
        for k in responses:
            if k.lower() in query.lower():
                return responses[k]
        return responses["default"]
    
    def _calculator(self, expression: str) -> str:
        try:
            result = eval(expression, {"__builtins__": {}}, {})
            return f"Result: {result}"
        except Exception as e:
            return f"Error: {str(e)}"
    
    def execute(self, decision: dict) -> dict:
        """Execute the tool and return an observation."""
        tool_name = decision["tool"]
        args = decision["args"]
        
        print(f"ACTION: Executing {tool_name}({args})")
        
        if tool_name not in self.registry:
            observation = f"Error: Tool '{tool_name}' not found in registry"
        else:
            observation = self.registry[tool_name](**args)
        
        print(f"ACTION: Got observation: {observation[:80]}")
        
        # Return as a tool result message (for adding to history)
        return {
            "role": "tool",
            "tool_call_id": decision["tool_call_id"],
            "content": observation
        }

# Test action module
action = ActionModule()
mock_decision = {
    "type": "tool_call",
    "tool": "web_search",
    "args": {"query": "AI agent frameworks 2025"},
    "tool_call_id": "call_test_001"
}

result = action.execute(mock_decision)
print(f"\nTool result message: {result}")

## Phase M: Memory — State Management

In [ ]:
class MemoryModule:
    """
    MEMORY: Manages the agent's state — what it remembers and retrieves.
    - Short-term: in-context conversation buffer
    - Long-term: (in production) vector store, SQL DB, knowledge graph
    """
    def __init__(self, max_stm_messages: int = 20):
        self.stm = []          # Short-term memory: conversation buffer
        self.ltm = {}          # Long-term memory: key-value store (simplified)
        self.max_stm = max_stm_messages
    
    def add_to_stm(self, message: dict):
        """Add message to short-term memory (conversation buffer)."""
        self.stm.append(message)
        # Sliding window: keep only recent messages
        if len(self.stm) > self.max_stm:
            removed = self.stm.pop(0)  # Remove oldest
            print(f"MEMORY: STM full — evicted oldest message (role={removed['role']})")
    
    def save_to_ltm(self, key: str, value: str):
        """Persist important facts to long-term memory."""
        self.ltm[key] = value
        print(f"MEMORY: Saved to LTM: '{key}'")
    
    def retrieve_from_ltm(self, key: str) -> str | None:
        """Retrieve a specific fact from long-term memory."""
        return self.ltm.get(key)
    
    def get_context(self) -> list[dict]:
        """Return current STM for use as context in next reasoning step."""
        return self.stm.copy()
    
    def show_state(self):
        print(f"Memory State:")
        print(f"  STM: {len(self.stm)} messages (max={self.max_stm})")
        print(f"  LTM: {len(self.ltm)} stored facts")
        for k, v in self.ltm.items():
            print(f"    '{k}': '{v[:50]}'")

# Demonstrate memory
memory = MemoryModule(max_stm_messages=5)

# Add messages to STM
for i in range(7):
    memory.add_to_stm({"role": "user" if i % 2 == 0 else "assistant", "content": f"Message {i+1}"})

# Save to LTM
memory.save_to_ltm("user_goal", "Research AI agent frameworks")
memory.save_to_ltm("preferred_format", "Bullet points, concise")

memory.show_state()

## Complete PRAM Agent — All 4 Phases Working Together

In [ ]:
class PRAMAgent:
    """
    Complete PRAM agent: all 4 phases working together in a loop.
    P → R → A → M → (repeat) → Terminate
    """
    
    def __init__(self):
        self.perception = PerceptionModule(
            "You are a research agent. Use web_search for current info. "
            "When you have enough info, answer directly without using tools."
        )
        self.reasoning = ReasoningModule()
        self.action = ActionModule()
        self.memory = MemoryModule(max_stm_messages=20)
    
    def run(self, user_input: str, max_steps: int = 5) -> str:
        print(f"{'='*60}")
        print(f"PRAM AGENT STARTING")
        print(f"Input: {user_input}")
        print(f"{'='*60}")
        
        for step in range(1, max_steps + 1):
            print(f"\n{'─'*50}")
            print(f"STEP {step}")
            print(f"{'─'*50}")
            
            # ── P: PERCEPTION ──────────────────────────────────────
            context = self.perception.assemble_context(
                user_input=user_input,
                history=self.memory.get_context()
            )
            
            # ── R: REASONING ───────────────────────────────────────
            decision = self.reasoning.reason(context)
            
            # ── TERMINATION CHECK ──────────────────────────────────
            if decision["type"] == "final_answer":
                print(f"\nTERMINATED: Agent completed in {step} step(s)")
                final = decision["content"]
                print(f"Final Answer: {final}")
                self.memory.show_state()
                return final
            
            # ── A: ACTION ──────────────────────────────────────────
            self.memory.add_to_stm(decision["raw_message"])  # Agent's tool call
            observation_msg = self.action.execute(decision)
            
            # ── M: MEMORY ──────────────────────────────────────────
            self.memory.add_to_stm(observation_msg)  # Tool result
            print(f"MEMORY: Added tool call + observation to STM")
            
            # Update user_input for next loop (context carries state)
            user_input = "Based on what you found, what is the final answer?"
        
        return "Max steps reached without completion."

# Run the full PRAM agent  (bug fixed: was PRMAgent → now PRAMAgent)
agent = PRAMAgent()
result = agent.run(
    user_input="What are the top 3 AI agent frameworks in 2025?",
    max_steps=5
)

## Key Takeaways

In [ ]:
rprint(Panel("""
[bold]The PRAM Loop — Key Insights[/bold]

P — PERCEPTION: Context assembly is an engineering challenge
    → You control what the LLM 'sees' — make it count

R — REASONING: The LLM makes two types of decisions
    → 'Use tool X' OR 'I have enough — final answer'
    → This binary is the control flow of the entire agent

A — ACTION: Tools are the hands of the agent
    → Design tool schemas carefully — LLMs follow descriptions
    → Always return structured, clean observations

M — MEMORY: State makes agents stateful, not stateless
    → STM = context window (finite, expensive)
    → LTM = external DB (infinite, retrieved by query)

Termination: ALWAYS define exit conditions
    → max_steps is your safety valve
""", title="PRAM Summary", border_style="cyan"))